# FDA Project — Improved Model
## S&P 500 Bubble Detection & Crisis Early-Warning System (v2)

Notebook นี้เป็นเวอร์ชันปรับปรุงจาก demo โดยแก้ข้อเสียที่พบจากผลรอบแรก และใช้ข้อมูลจริงทั้ง train และ test

| ข้อเสียที่พบใน demo | สิ่งที่แก้ใน v2 | อ้างอิง |
|---|---|---|
| Train แค่ 1997–2008 → เห็นวิกฤตใหญ่แค่ 2 ครั้ง | **Expanding walk-forward** + เพิ่ม fold วิกฤต 2008, 2011, 2015, 2018 (รวม 6 fold) | — |
| Sector feature ทำให้ข้อมูลเริ่มกลางปี 2000 | ตัด sector feature ออก → train เริ่ม 1997 จริง | — |
| Weekly price BSADF เป็น noise และ flag ช่วงตลาดร่วงเป็น bubble | Bubble signal ใช้ **monthly P/D GSADF** และ **แยก bubble กับ crash** ตามทิศทาง | Phillips & Shi (2020); Başoğlu Kabran & Ünlü (2021) |
| ADF lag 0 + Monte Carlo แบบ homoskedastic | เลือก lag ด้วย **BIC** + **wild bootstrap** ที่คุม multiplicity (B = 999) | Phillips & Shi (2020) |
| Label ปน "วิกฤตที่เกิดแล้ว" → เตือนผิดหลังตลาดร่วง (AUC < 0.5 ใน COVID) | **Mask** สัปดาห์ที่อยู่ใน crash regime หรือ drawdown ≥ 10% ออกจากทั้ง train และ test | Phillips & Shi (2020) Sec. 4 |
| Feature เป็นค่าระดับ, ซ้ำกันสูง (VIX / STLFSI4 / HMM ≈ 0.9) | ใช้ **ค่าการเปลี่ยนแปลง (Δ)**, ตัดตัวซ้ำ, เพิ่ม **ดอกเบี้ย / เงินเฟ้อ / ว่างงาน** | Başoğlu Kabran & Ünlü (2021) |
| ไม่มีตัวจับความเสี่ยงด้านเครดิตที่พุ่งขึ้น | **PSY บน credit spread** (BAA–AAA) | Phillips & Shi (2020) Example 2 |
| Threshold 0.5 → Recall เกือบ 0; recall target → เตือนทุกสัปดาห์ (Kappa = 0) | **Alert budget**: เตือน 20% ของสัปดาห์ที่เสี่ยงที่สุดตาม OOF score ใน training | — |
| AUC ตัวเดียวแยกไม่ได้ว่า "สุ่ม" หรือ "กลับทิศ" | **AUC + 95% block-bootstrap CI** เทียบกับ 0.5 | — |
| ไม่รู้ว่า skill มาจากความต่อเนื่องของวิกฤตเท่าไร | **ตาราง Results: ไม่ตัด vs ตัด** สัปดาห์วิกฤต + paired bootstrap (Part 6.1) | Phillips & Shi (2020) |
| ผล replication ต่างจาก paper ช่วง 1998–2000 | Validation เทียบ **BIC lag vs lag = 0** (`FORCE_LAG`) | Phillips & Shi (2020) |
| RF 15 feature กับวิกฤตราว 15 ครั้ง | เพิ่ม **Logit5** (5 feature ที่มีทฤษฎีรองรับ) | Estrella & Mishkin (1998); Danielsson et al. (2018) |
| Horizon สั้นเกินไปสำหรับ yield curve / valuation | เพิ่ม **12M** horizon + feature **`Low_Vol_Weeks`** (volatility paradox) | Brunnermeier & Sannikov (2014); Danielsson et al. (2018) |
| ไม่รู้ว่าดีกว่ากฎง่ายๆ ไหม | **Benchmark**: VIX > 25, yield curve inverted, P/D bubble flag | — |
| ผลต่างระหว่างโมเดลไม่มีการทดสอบ | **Block bootstrap** บน predictions รวมทุก fold | Gu (2022) |
| HMM fit ครั้งเดียวถึงปี 2008 | **Refit HMM ทุก fold** ด้วยข้อมูลถึง train end เท่านั้น (filtered probability) | Gu (2022) |
| COVID พยากรณ์ล่วงหน้าไม่ได้ | ระบบมี **2 output**: early warning + **real-time crash detection** | Phillips & Shi (2020) |

**ส่วนที่ตัดออกจาก demo:** sector decomposition (ยังอยู่ใน demo notebook), LPPLS extension และ feature ที่ซ้ำกัน

## Part 0 — Set up

In [ ]:
!pip install -q hmmlearn shap yfinance xlrd

In [ ]:
# — Data handling ———————————————————————————————————————————
import numpy as np
import pandas as pd
import io, os, requests

# — Visualization ———————————————————————————————————————————
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns

# — Statistics / models —————————————————————————————————————
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, recall_score,
                             precision_score, cohen_kappa_score)
from hmmlearn.hmm import GaussianHMM
import shap

import warnings
warnings.filterwarnings("ignore")
print("Libraries ready.")

In [ ]:
# — Global settings —————————————————————————————————————————
SPX_START    = "1980-01-01"      # longer price history for the weekly PSY crash detector
START        = "1990-01-01"      # VIX starts in 1990
END          = "2024-12-31"
TRADING_DAYS = 252
WINDOW       = 20
FREQ         = "W-FRI"
SEED         = 42

# — Labels —
CRISIS_DD    = -0.10             # crisis = S&P 500 falls >= 10% within the next H weeks
IN_CRISIS_DD = -0.10             # mask weeks already >= 10% below the 52-week high
HORIZONS     = {"1M": 4, "3M": 13, "6M": 26, "12M": 52}   # 12M: yield curve / valuation lead ~12-18 months

# — PSY / GSADF —
P_MAX        = 6                 # max ADF lag for BIC selection (Phillips & Shi, 2020)
N_BOOT       = 999               # wild-bootstrap replications
TB_MONTHS    = 24                # family-wise size control window (monthly series)
TB_WEEKS     = 52                # family-wise size control window (weekly series)
PD_PUB_LAG   = 1                 # months of publication lag for dividend data
MACRO_LAG    = 1                 # months of publication lag for CPI / unemployment
DIR_LB_M     = 6                 # look-back (months) to classify bubble vs crash direction
DIR_LB_W     = 13                # look-back (weeks)
FORCE_LAG    = None              # None = BIC lag (paper); 0 = force ADF lag 0 everywhere

# — Walk-forward design (expanding window) —
TRAIN_START  = "1997-01-01"
FOLDS = {
    "GFC-2008":   {"train_end": "2006-12-31", "test": ("2007-01-01", "2009-12-31")},
    "Euro-2011":  {"train_end": "2010-12-31", "test": ("2011-01-01", "2012-12-31")},
    "China-2015": {"train_end": "2014-12-31", "test": ("2015-01-01", "2016-12-31")},
    "Q4-2018":    {"train_end": "2017-12-31", "test": ("2018-01-01", "2018-12-31")},
    "COVID-2020": {"train_end": "2018-12-31", "test": ("2019-01-01", "2020-12-31")},
    "Rates-2022": {"train_end": "2020-12-31", "test": ("2021-01-01", "2023-06-30")},
}
MAIN_FOLDS   = ["COVID-2020", "Rates-2022"]          # the two test periods in the proposal
PROPOSAL_TRAIN_END = "2008-12-31"                    # fixed-window scheme in the proposal

# Bootstrap critical values and ADF lags use data up to the EARLIEST train end only,
# so no fold ever sees information from its own test period.
CV_FIT_END   = min(f["train_end"] for f in FOLDS.values())

# — Volatility paradox feature —
LOWVOL_SMOOTH   = 13             # weeks: smoothing of 20-day realised volatility
LOWVOL_LOOKBACK = 520            # weeks: trailing window for the "normal" volatility median

# — Classification —
ALERT_BUDGET  = 0.20             # alert on the riskiest 20% of weeks (OOF scores in training)
N_BOOT_AUC    = 1000             # block-bootstrap replications for AUC confidence intervals
ALERT_K       = 2                # consecutive weeks above threshold to raise an alert

colour_map = {"Calm": "#2ecc71", "Volatile": "#f4a261", "Crisis": "#e74c3c"}
print(f"Settings ready. CV / lag selection fitted on data up to {CV_FIT_END}.")

## Part 1 — Data

In [ ]:
# — Download S&P 500 (from 1980) and VIX (from 1990) ————————
import yfinance as yf

def yf_close(ticker, start):
    raw = yf.download(ticker, start=start, end=END, auto_adjust=True, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    return raw["Close"].rename(ticker)

spx = yf_close("^GSPC", SPX_START)
vix = yf_close("^VIX", START)

daily = pd.DataFrame({"SPX": spx}).join(vix.rename("VIX"))
daily.index.name = "Date"
daily = daily.ffill()

assert daily.index.is_monotonic_increasing, "Dates are not sorted."
assert not daily.index.has_duplicates, "Duplicate dates found."

print("=" * 60)
print(f"S&P 500 : {spx.index.min().date()} -> {spx.index.max().date()} ({len(spx):,} days)")
print(f"VIX     : {vix.index.min().date()} -> {vix.index.max().date()} ({len(vix):,} days)")
print("=" * 60)

In [ ]:
# — Macro-financial series from FRED ——————————————————————————
def load_fred(series_id):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    s = pd.read_csv(url, index_col=0, parse_dates=True, na_values=".")
    return s.iloc[:, 0].astype(float).rename(series_id).dropna()

def align_to(index, s):
    # forward-fill a series onto another index using only past values
    return s.reindex(s.index.union(index)).ffill().reindex(index)

def monthly_available(s, lag):
    # month-start stamped monthly data -> usable at the end of month M + lag months
    s = s.copy()
    idx = s.index + pd.offsets.MonthEnd(1)
    for _ in range(lag):
        idx = idx + pd.offsets.MonthEnd(1)
    s.index = idx
    return s

FRED_IDS = ["T10Y2Y", "DBAA", "DAAA", "DFF", "CPIAUCSL", "UNRATE", "USREC"]
fred = {sid: load_fred(sid) for sid in FRED_IDS}

print("=" * 60)
for sid, s in fred.items():
    print(f"{sid:9s}: {s.index.min().date()} -> {s.index.max().date()}  ({len(s):,} obs)")
print("=" * 60)

In [ ]:
# — Shiller data: price-dividend ratio and CAPE ———————————————
SHILLER_URLS = [
    "https://img1.wsimg.com/blobby/go/e5e77e0b-59d1-44d9-ab25-4763ac982e53/downloads/ie_data.xls",
    "http://www.econ.yale.edu/~shiller/data/ie_data.xls",
]
SHILLER_LOCAL = "ie_data.xls"      # upload this file to Colab if both URLs fail

def parse_shiller(src):
    x = pd.read_excel(src, sheet_name="Data", skiprows=7)
    x = x[["Date", "P", "D", "CAPE"]].apply(pd.to_numeric, errors="coerce")
    x = x.dropna(subset=["Date", "P"])
    yr = x["Date"].astype(int)
    mo = ((x["Date"] - yr) * 100).round().astype(int)
    x.index = pd.to_datetime(dict(year=yr, month=mo, day=1))
    return (x["P"] / x["D"]).dropna().rename("PD"), x["CAPE"].dropna().rename("CAPE")

def multpl_table(path):
    url = f"https://www.multpl.com/{path}/table/by-month"
    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
    tbl = pd.read_html(io.StringIO(html))[0]
    tbl.columns = ["Date", "Value"]
    tbl["Date"] = pd.to_datetime(tbl["Date"])
    tbl = tbl[tbl["Date"].dt.day == 1]                          # drop intra-month estimate
    v = pd.to_numeric(tbl["Value"].astype(str).str.extract(r"([\d\.]+)")[0], errors="coerce")
    return v.set_axis(tbl["Date"]).sort_index().dropna()

def load_shiller():
    sources = ([SHILLER_LOCAL] if os.path.exists(SHILLER_LOCAL) else []) + SHILLER_URLS
    for src in sources:
        try:
            pdr, cape = parse_shiller(src)
            print(f"Shiller source: {src}")
            return pdr, cape
        except Exception as e:
            print(f"Shiller source failed ({type(e).__name__}) -> next")
    print("Using multpl.com (dividend yield -> P/D, Shiller PE -> CAPE)")
    pdr  = (100 / multpl_table("s-p-500-dividend-yield")).rename("PD")
    cape = multpl_table("shiller-pe").rename("CAPE")
    return pdr, cape

pd_ratio, cape = load_shiller()

print("=" * 60)
print(f"P/D  : {pd_ratio.index.min().date()} -> {pd_ratio.index.max().date()} ({len(pd_ratio):,} months)")
print(f"CAPE : {cape.index.min().date()} -> {cape.index.max().date()} ({len(cape):,} months)")
print("=" * 60)

## Part 2 — PSY Engine (GSADF with BIC lag + wild bootstrap)

ทำตาม Phillips & Shi (2020):
- ADF regression: $\Delta y_t = \mu + \rho y_{t-1} + \sum_{j=1}^{p} \phi_j \Delta y_{t-j} + v_t$
- **BSADF$_t$** = sup ของ ADF t-stat ของ $\rho$ จากทุก window ที่จบที่ $t$ (real-time)
- **Lag $p$** เลือกด้วย BIC (สูงสุด 6) จากข้อมูลช่วง training แล้วใช้ค่าเดียวทุก window
  *(paper เลือก lag ใหม่ทุก sub-window — เราใช้ค่าคงที่เพื่อให้คำนวณแบบ vectorised ได้)*
- **Wild bootstrap (Step 1–5 ใน paper)**: fit null model ($\rho = 0$) → สร้าง series ยาว $\tau_0 + \tau_b - 1$
  ด้วย residual × N(0,1) → เอา max ของ BSADF ในช่วง $\tau_b$ → quantile 95% = critical value (คุม heteroskedasticity + multiplicity)
- **Bubble vs Crash** (Sec. 4): PSY ปฏิเสธ H0 ได้ทั้งตอน explosive ขาขึ้นและตอน crash → แยกด้วยทิศทางของ series ย้อนหลัง

BSADF คำนวณผ่าน cumulative sums ของ $X'X$, $X'z$ ทำให้ทุก window คำนวณได้ใน O(1) และตรวจสอบแล้วว่าตรงกับ OLS ทีละ window

In [ ]:
# — Vectorised BSADF with p lags ——————————————————————————————
def adf_design(Y, p):
    N, T = Y.shape
    dY = np.diff(Y, axis=1)
    R = T - 1 - p
    z = dY[:, p:]                                        # Δy_t
    cols = [np.ones((N, R)), Y[:, p:T - 1]]              # 1, y_{t-1}
    for j in range(1, p + 1):
        cols.append(dY[:, p - j:T - 1 - j])              # Δy_{t-j}
    return z, np.stack(cols, axis=2)

def bsadf(Y, w0, p=0):
    # Y: (N, T) log series. Returns (N, T) BSADF; NaN before the first full window.
    Y = np.atleast_2d(np.asarray(Y, dtype=float))
    Y = Y - Y[:, :1]
    N, T = Y.shape
    z, X = adf_design(Y, p)
    R, k = z.shape[1], X.shape[2]
    m = w0 - 1 - p                                       # min regression rows per window

    def csum(a):
        return np.concatenate([np.zeros((N, 1) + a.shape[2:]), np.cumsum(a, axis=1)], axis=1)

    Cxx = csum(np.einsum("nri,nrj->nrij", X, X))
    Cxz = csum(X * z[:, :, None])
    Czz = csum(z * z)
    e1 = np.zeros((k, 1)); e1[1, 0] = 1.0
    out = np.full((N, T), np.nan)

    for j in range(m - 1, R):
        starts = np.arange(0, j - m + 2)
        n   = (j + 1 - starts).astype(float)
        XtX = Cxx[:, [j + 1]] - Cxx[:, starts]
        Xtz = Cxz[:, [j + 1]] - Cxz[:, starts]
        ztz = Czz[:, [j + 1]] - Czz[:, starts]
        rhs = np.concatenate([Xtz[..., None],
                              np.broadcast_to(e1, XtX.shape[:-1] + (1,))], axis=-1)
        sol = np.linalg.solve(XtX, rhs)
        b, inv11 = sol[..., 0], sol[..., 1, 1]
        ssr = np.maximum(ztz - np.einsum("nsk,nsk->ns", b, Xtz), 1e-12)
        t_rho = b[..., 1] / np.sqrt(ssr / (n - k) * inv11)
        out[:, j + 1 + p] = np.nanmax(t_rho, axis=1)
    return out

def select_lag_bic(y, p_max=P_MAX):
    # BIC of the full-sample ADF regression; common sample across p for comparability
    y = np.asarray(y, dtype=float)
    bics = {}
    for p in range(p_max + 1):
        z, X = adf_design(y[None, :], p)
        z, X = z[0, p_max - p:], X[0, p_max - p:]
        b, *_ = np.linalg.lstsq(X, z, rcond=None)
        ssr = np.sum((z - X @ b) ** 2)
        n = len(z)
        bics[p] = n * np.log(ssr / n) + X.shape[1] * np.log(n)
    return min(bics, key=bics.get)

def wild_bootstrap_cv(y, w0, p, tb, n_boot=N_BOOT, q=0.95, seed=SEED):
    # Phillips & Shi (2020), Steps 1-5
    rng = np.random.default_rng(seed)
    y  = np.asarray(y, dtype=float)
    dy = np.diff(y)
    # Step 1: null model (rho = 0): dy_t = mu + sum phi_j dy_{t-j} + e_t
    Xn = np.column_stack([np.ones(len(dy) - p)] + [dy[p - j:len(dy) - j] for j in range(1, p + 1)])
    coef, *_ = np.linalg.lstsq(Xn, dy[p:], rcond=None)
    resid = dy[p:] - Xn @ coef
    # Step 2: bootstrap samples of length tau0 + tau_b - 1
    T_b = w0 + tb - 1
    dyb = np.zeros((n_boot, T_b - 1))
    dyb[:, :p] = dy[:p]
    w   = rng.standard_normal((n_boot, T_b - 1))
    idx = rng.integers(0, len(resid), size=(n_boot, T_b - 1))
    for t in range(p, T_b - 1):
        ar = sum(coef[j] * dyb[:, t - j] for j in range(1, p + 1)) if p > 0 else 0.0
        dyb[:, t] = coef[0] + ar + w[:, t] * resid[idx[:, t]]
    Yb = y[0] + np.concatenate([np.zeros((n_boot, 1)), np.cumsum(dyb, axis=1)], axis=1)
    # Step 3-5: max BSADF over the tau_b window, 95% quantile
    M = np.nanmax(bsadf(Yb, w0, p), axis=1)
    return np.quantile(M, q)

def run_psy(series, tb, lb_dir, fit_end=CV_FIT_END, n_boot=N_BOOT, lag=None):
    # lag=None -> BIC selection (Phillips & Shi, 2020); lag=0 -> force ADF lag 0
    # Full pipeline on one series (log scale). Returns DataFrame + info dict.
    y_all = np.log(series.values)
    T  = len(y_all)
    r0 = 0.01 + 1.8 / np.sqrt(T)
    w0 = int(np.floor(r0 * T))
    y_fit = np.log(series.loc[:fit_end].values)
    p  = select_lag_bic(y_fit) if lag is None else int(lag)
    cv = wild_bootstrap_cv(y_fit, w0, p, tb, n_boot=n_boot)
    out = pd.DataFrame(index=series.index)
    out["BSADF"] = bsadf(y_all, w0, p)[0]
    out["Gap"]   = out["BSADF"] - cv
    flag = (out["Gap"] > 0).astype(float).where(out["BSADF"].notna())
    chg  = pd.Series(y_all, index=series.index).diff(lb_dir)
    out["Flag"]   = flag
    out["Up"]     = (flag.eq(1) & chg.gt(0)).astype(float).where(flag.notna())
    out["Crash"]  = (flag.eq(1) & chg.le(0)).astype(float).where(flag.notna())
    info = {"T": T, "w0": w0, "lag": p, "cv": cv, "gsadf": np.nanmax(out["BSADF"])}
    return out, info

def episodes(flag, min_len=1, fmt="%Y-%m"):
    flag = flag.fillna(0).astype(bool)
    grp = (flag != flag.shift()).cumsum()
    rows = [(g.index[0].strftime(fmt), g.index[-1].strftime(fmt), len(g))
            for _, g in flag.groupby(grp) if g.iloc[0] and len(g) >= min_len]
    return pd.DataFrame(rows, columns=["Start", "End", "Periods"])

print("PSY engine ready.")

### 2.1 Validation — reproduce Phillips & Shi (2020), S&P 500 example

paper รัน P/D รายเดือน ม.ค. 1973 – ก.ค. 2018, BIC lag ≤ 6, $\tau_b$ = 24, B = 999
รันทั้ง **BIC lag** และ **บังคับ lag = 0** แล้วดูว่าแบบไหน match กับ paper มากกว่า โดยเฉพาะช่วง 1998–2000
(lag ที่ใช้ทั้ง notebook ตั้งได้ที่ `FORCE_LAG` ใน Global settings)
เราใช้ P/D จาก Shiller (paper ใช้ Datastream) จึงคาดว่าจะ **ใกล้เคียงแต่ไม่เหมือน 100%**

In [ ]:
# — Reproduce the S&P 500 example of Phillips & Shi (2020) ————
PAPER_EPISODES = pd.DataFrame({
    "Start": ["1986-05", "1987-07", "1996-01", "1996-05", "1996-11", "1997-04", "1998-09", "2000-12", "2008-10"],
    "End":   ["1986-06", "1987-08", "1996-01", "1996-05", "1997-02", "1998-07", "2000-10", "2001-01", "2009-02"],
})

pd_val = pd_ratio.loc["1973-01-01":"2018-07-31"]
paper_months = pd.Index([])
for _, r in PAPER_EPISODES.iterrows():
    paper_months = paper_months.union(pd.date_range(r["Start"], r["End"], freq="MS"))
WIN = ("1998-01-01", "2000-12-31")                     # dot-com window
in_win = lambda idx: idx[(idx >= WIN[0]) & (idx <= WIN[1])]

# Paper's setup: residuals from the full sample (this is a replication, not a forecast)
val_runs, rows = {}, []
for lag_name, lag in [("BIC lag", None), ("Lag = 0", 0)]:
    v, info = run_psy(pd_val, tb=TB_MONTHS, lb_dir=DIR_LB_M, fit_end="2018-07-31", lag=lag)
    ours_m = v.index[v["Flag"].eq(1)]
    pw, ow = in_win(paper_months), in_win(ours_m)
    val_runs[lag_name] = (v, info)
    rows.append({"Setting": lag_name, "Lag used": info["lag"],
                 "GSADF": info["gsadf"], "CV (95%)": info["cv"],
                 "Paper months matched": paper_months.isin(ours_m).mean(),
                 "Extra months (not in paper)": int((~ours_m.isin(paper_months)).sum()),
                 "1998–2000 matched": pw.isin(ow).mean(),
                 "1998–2000 extra": int((~ow.isin(pw)).sum()),
                 "Jaccard (all)": len(paper_months.intersection(ours_m)) / max(len(paper_months.union(ours_m)), 1)})
val_cmp = pd.DataFrame(rows).set_index("Setting")

# Keep the setting used in the rest of the notebook for the summary
val, val_info = val_runs["Lag = 0" if FORCE_LAG == 0 else "BIC lag"]
ours = episodes(val["Flag"])
hit = val_cmp.loc["Lag = 0" if FORCE_LAG == 0 else "BIC lag", "Paper months matched"]

print("=== Replication of Phillips & Shi (2020): BIC lag vs forced lag 0 ===")
display(val_cmp.round(3))
print("Paper (Phillips & Shi, 2020):")
display(PAPER_EPISODES)
for lag_name, (v, info) in val_runs.items():
    print(f"This notebook — {lag_name} (lag {info['lag']}):")
    display(episodes(v["Flag"]))

## Part 3 — Layer 1: PSY Signals

| Series | Frequency | ใช้ทำอะไร |
|---|---|---|
| S&P 500 **P/D ratio** (1871–) | Monthly | **Bubble signal** (leading): `PD_BSADF_Gap`, `PD_Bubble_Up`, `PD_Bubble_Months` |
| **Credit spread** BAA–AAA (1986–) | Weekly | **Credit stress ballooning**: `Credit_BSADF_Gap`, `Credit_Up` |
| S&P 500 **price** (1980–) | Weekly | **Real-time crash detector** (`Crash_Flag`) → ใช้ mask label และเป็น output ที่ 2 ของระบบ |

In [ ]:
# — 3.1 Monthly P/D bubble signal —————————————————————————————
pd_sig, pd_info = run_psy(pd_ratio, tb=TB_MONTHS, lb_dir=DIR_LB_M, lag=FORCE_LAG)

run_id = (pd_sig["Up"] != pd_sig["Up"].shift()).cumsum()
pd_sig["Bubble_Months"] = pd_sig.groupby(run_id).cumcount().add(1) * pd_sig["Up"]

print("=" * 60)
print(f"P/D months: {pd_info['T']} | min window: {pd_info['w0']} | BIC lag: {pd_info['lag']}")
print(f"GSADF = {pd_info['gsadf']:.3f} | wild-bootstrap CV = {pd_info['cv']:.3f}")
print(f"Months flagged: bubble {int(pd_sig['Up'].sum())} | crash {int(pd_sig['Crash'].sum())}")
print("=" * 60)
ep = episodes(pd_sig["Up"], min_len=3)
print("=== P/D bubble episodes (>= 3 months) ===")
display(ep.rename(columns={"Periods": "Months"}))

In [ ]:
# — 3.2 Weekly credit-spread PSY ——————————————————————————————
credit_d  = (fred["DBAA"] - fred["DAAA"]).dropna()
credit_w  = credit_d.resample(FREQ).last().dropna()
credit_w  = credit_w[credit_w > 0]
cr_sig, cr_info = run_psy(credit_w, tb=TB_WEEKS, lb_dir=DIR_LB_W, lag=FORCE_LAG)

print("=" * 60)
print(f"Credit weeks: {cr_info['T']} | min window: {cr_info['w0']} | BIC lag: {cr_info['lag']}")
print(f"GSADF = {cr_info['gsadf']:.3f} | wild-bootstrap CV = {cr_info['cv']:.3f}")
print(f"Weeks with explosive widening (Credit_Up): {int(cr_sig['Up'].sum())}")
print("=" * 60)

In [ ]:
# — 3.3 Weekly S&P 500 price PSY -> real-time crash detector ———
spx_w = daily["SPX"].resample(FREQ).last().dropna()
px_sig, px_info = run_psy(spx_w, tb=TB_WEEKS, lb_dir=DIR_LB_W, lag=FORCE_LAG)

print("=" * 60)
print(f"Price weeks: {px_info['T']} | min window: {px_info['w0']} | BIC lag: {px_info['lag']}")
print(f"GSADF = {px_info['gsadf']:.3f} | wild-bootstrap CV = {px_info['cv']:.3f}")
print(f"Weeks flagged: up {int(px_sig['Up'].sum())} | crash {int(px_sig['Crash'].sum())}")
print("=" * 60)
ep = episodes(px_sig["Crash"], min_len=2, fmt="%Y-%m-%d")
print("=== Crash episodes detected on weekly prices (>= 2 weeks) ===")
display(ep.rename(columns={"Periods": "Weeks"}))

In [ ]:
# — Visualise the three PSY signals ———————————————————————————
fig, axes = plt.subplots(3, 1, figsize=(13, 11))
fig.suptitle("Layer 1 — PSY signals (wild-bootstrap critical values)", fontsize=13, fontweight="bold")

panels = [
    (axes[0], pd_ratio, pd_sig, "S&P 500 P/D ratio (monthly)", "#9b59b6", "Bubble (up)"),
    (axes[1], credit_w, cr_sig, "BAA–AAA credit spread (weekly)", "#f4a261", "Explosive widening"),
    (axes[2], spx_w, px_sig, "S&P 500 price (weekly, log)", "#2ecc71", "Explosive up"),
]
for ax, s, sig, title, col_up, lab_up in panels:
    ax.plot(s.index, s.values, color="#1f4e79", lw=1)
    ax.fill_between(sig.index, 0, 1, where=sig["Up"].eq(1), color=col_up, alpha=0.35,
                    transform=ax.get_xaxis_transform(), label=lab_up)
    ax.fill_between(sig.index, 0, 1, where=sig["Crash"].eq(1), color="#e74c3c", alpha=0.35,
                    transform=ax.get_xaxis_transform(), label="Crash / collapse")
    ax.set_title(title, fontsize=11)
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
axes[2].set_yscale("log")
axes[0].set_xlim(pd.Timestamp("1960-01-01"), pd_ratio.index.max())
plt.tight_layout()
plt.show()

## Part 4 — Weekly Features & Labels

**Features** (ทุกตัวใช้ข้อมูล ณ สัปดาห์นั้นหรือก่อนหน้า; ข้อมูลรายเดือนมี publication lag):

| Group | Feature | หมายเหตุ |
|---|---|---|
| Bubble (P/D) | `PD_BSADF_Gap`, `PD_Bubble_Up`, `PD_Bubble_Months` | lag = สิ้นเดือน + 1 เดือน |
| Credit PSY | `Credit_BSADF_Gap`, `Credit_Up` | |
| HMM | `HMM_P_Crisis`, `HMM_P_Volatile` | refit ทุก fold (Part 5) |
| Market | `VIX_Gap` (VIX − ค่าเฉลี่ย 52 สัปดาห์), `Credit_Spread_Chg13w` | แทนค่าระดับ VIX / STLFSI4 ที่ซ้ำกัน |
| Volatility paradox | `Low_Vol_Weeks` = จำนวนสัปดาห์ติดกันที่ realised vol (เฉลี่ย 13 สัปดาห์) ต่ำกว่า median 10 ปีย้อนหลัง | Danielsson, Valenzuela & Zer (2018) |
| Rates & macro | `T10Y2Y`, `T10Y2Y_Chg13w`, `FedFunds_Chg26w`, `CPI_YoY`, `Unemp_Gap`, `CAPE_Pctl` | `Unemp_Gap` = Sahm-rule gap |

**Label** `Crisis_H` = 1 ถ้า S&P 500 ลง ≥ 10% ภายใน H สัปดาห์ (H = 1M, 3M, 6M, **12M**)
**Logit5** ใช้ 5 feature ที่มีทฤษฎีรองรับ: `T10Y2Y`, `CAPE_Pctl`, `Credit_Spread_Chg13w`, `PD_BSADF_Gap`, `Low_Vol_Weeks`
**Mask**: สัปดาห์ที่ (ก) ต่ำกว่า 52-week high ≥ 10% หรือ (ข) weekly PSY บอกว่าอยู่ใน crash regime → ไม่ใช้ทั้งตอน train และ test
เพราะเป็นช่วงที่วิกฤต "เกิดไปแล้ว" ไม่ใช่ช่วงที่ต้องเตือนล่วงหน้า

In [ ]:
# — Daily market and macro inputs ——————————————————————————————
daily["T10Y2Y"]   = align_to(daily.index, fred["T10Y2Y"])
daily["Credit"]   = align_to(daily.index, credit_d)
daily["DFF"]      = align_to(daily.index, fred["DFF"])
daily["USREC"]    = align_to(daily.index, fred["USREC"])

cpi_yoy   = fred["CPIAUCSL"].pct_change(12) * 100
u3        = fred["UNRATE"].rolling(3).mean()
unemp_gap = u3 - u3.shift(1).rolling(12).min()                 # Sahm-rule gap
daily["RV20"] = np.log(daily["SPX"]).diff().rolling(WINDOW).std() * np.sqrt(TRADING_DAYS)
daily["CPI_YoY"]   = align_to(daily.index, monthly_available(cpi_yoy.dropna(), MACRO_LAG))
daily["Unemp_Gap"] = align_to(daily.index, monthly_available(unemp_gap.dropna(), MACRO_LAG))

cape_pctl = cape.expanding(min_periods=120).apply(lambda s: (s <= s[-1]).mean(), raw=True)
daily["CAPE_Pctl"] = align_to(daily.index, monthly_available(cape_pctl.dropna(), PD_PUB_LAG))

daily["DD_52w"] = daily["SPX"] / daily["SPX"].rolling(TRADING_DAYS, min_periods=60).max() - 1

# — Forward-looking crisis labels (daily, then sampled weekly) —
for name, h in HORIZONS.items():
    h_days = h * 5
    fut_min = daily["SPX"][::-1].rolling(h_days, min_periods=h_days).min()[::-1].shift(-1)
    fwd_dd  = fut_min / daily["SPX"] - 1
    daily[f"Crisis_{name}"] = np.where(fwd_dd.isna(), np.nan, (fwd_dd <= CRISIS_DD).astype(float))

print("Daily inputs ready.")

In [ ]:
# — Weekly modelling table (fold-independent part) ————————————
base_full = daily.resample(FREQ).last()

# Volatility paradox (Brunnermeier & Sannikov, 2014; Danielsson, Valenzuela & Zer, 2018):
# long stretches of low volatility encourage leverage and risk-taking that later unwinds.
# Low_Vol_Weeks = consecutive weeks with smoothed realised vol below its trailing 10-year median.
vol_s   = base_full["RV20"].rolling(LOWVOL_SMOOTH).mean()
vol_med = vol_s.rolling(LOWVOL_LOOKBACK, min_periods=260).median()
low     = (vol_s < vol_med).astype(int)
base_full["Low_Vol_Weeks"] = (low.groupby((low != low.shift()).cumsum()).cumsum() * low).where(vol_med.notna())

base = base_full.loc[START:].copy()

base["VIX_Gap"]              = base["VIX"] - base["VIX"].rolling(52).mean()
base["Credit_Spread_Chg13w"] = base["Credit"].diff(13)
base["T10Y2Y_Chg13w"]        = base["T10Y2Y"].diff(13)
base["FedFunds_Chg26w"]      = base["DFF"].diff(26)

# Monthly P/D signals with publication lag
for src, dst in [("Gap", "PD_BSADF_Gap"), ("Up", "PD_Bubble_Up"), ("Bubble_Months", "PD_Bubble_Months")]:
    base[dst] = align_to(base.index, monthly_available(pd_sig[src].dropna(), PD_PUB_LAG))

# Weekly PSY signals (computed on Friday closes -> no extra lag)
base["Credit_BSADF_Gap"] = cr_sig["Gap"].reindex(base.index)
base["Credit_Up"]        = cr_sig["Up"].reindex(base.index)
base["Crash_Flag"]       = px_sig["Crash"].reindex(base.index)
base["Price_BSADF_Gap"]  = px_sig["Gap"].reindex(base.index)     # ablation only
base["Price_Bubble_Up"]  = px_sig["Up"].reindex(base.index)      # ablation only

# Mask weeks where the crisis has already started
base["In_Crisis"] = (base["DD_52w"].le(IN_CRISIS_DD) | base["Crash_Flag"].eq(1)).astype(int)
for name in HORIZONS:
    base[f"Label_{name}"] = base[f"Crisis_{name}"].where(base["In_Crisis"].eq(0))

PSY_FEATS    = ["PD_BSADF_Gap", "PD_Bubble_Up", "PD_Bubble_Months", "Credit_BSADF_Gap", "Credit_Up"]
HMM_FEATS    = ["HMM_P_Crisis", "HMM_P_Volatile"]
MARKET_FEATS = ["VIX_Gap", "Credit_Spread_Chg13w", "Low_Vol_Weeks"]
MACRO_FEATS  = ["T10Y2Y", "T10Y2Y_Chg13w", "FedFunds_Chg26w", "CPI_YoY", "Unemp_Gap", "CAPE_Pctl"]
FEATURES     = PSY_FEATS + HMM_FEATS + MARKET_FEATS + MACRO_FEATS
PRICE_PSY    = ["Price_BSADF_Gap", "Price_Bubble_Up"]

# Parsimonious, theory-based set for "Logit5" (about 15 crisis events -> few parameters)
SMALL_FEATS  = ["T10Y2Y",                # yield curve (Estrella & Mishkin, 1998)
                "CAPE_Pctl",             # valuation (Campbell & Shiller, 1998)
                "Credit_Spread_Chg13w",  # credit conditions (Gilchrist & Zakrajšek, 2012)
                "PD_BSADF_Gap",          # bubble signal (Phillips & Shi, 2020)
                "Low_Vol_Weeks"]         # volatility paradox (Danielsson et al., 2018)

print("=" * 60)
print(f"Weekly rows    : {len(base):,} ({base.index.min().date()} -> {base.index.max().date()})")
print(f"Features       : {len(FEATURES)}")
print(f"Masked weeks   : {int(base['In_Crisis'].sum())} ({base['In_Crisis'].mean():.1%})")
print("=" * 60)
rows = []
for name in HORIZONS:
    rows.append({"Horizon": name,
                 "Raw base rate": base[f"Crisis_{name}"].loc[TRAIN_START:].mean(),
                 "After mask": base[f"Label_{name}"].loc[TRAIN_START:].mean(),
                 "Usable weeks": int(base[f"Label_{name}"].loc[TRAIN_START:].notna().sum())})
print("=== Label base rates (1997 onwards) ===")
display(pd.DataFrame(rows).set_index("Horizon").round(3))

In [ ]:
# — Redundancy check on non-HMM features ——————————————————————
corr = base.loc[TRAIN_START:, [f for f in FEATURES if f not in HMM_FEATS]].corr()
pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), 1)).stack()
         .rename("corr").reset_index().rename(columns={"level_0": "A", "level_1": "B"}))
high = pairs[pairs["corr"].abs() > 0.8]
print("=== Feature pairs with |corr| > 0.8 ===")
display(high if len(high) else "None — no highly redundant pairs.")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, annot_kws={"size": 7})
ax.set_title("Feature correlation (1997 onwards, excluding HMM)", fontsize=12)
plt.tight_layout()
plt.show()

## Part 5 — Layer 2: HMM refitted for every fold

ทุก fold: fit scaler + HMM ด้วยข้อมูลรายวันตั้งแต่ 1990 ถึง `train_end` ของ fold นั้นเท่านั้น
แล้วคำนวณ **filtered probability** $P(S_t \mid y_{1:t})$ ไปข้างหน้า (ไม่มี look-ahead) — แนวคิดจาก Gu (2022)

In [ ]:
# — HMM helpers ————————————————————————————————————————————————
HMM_INPUTS = ["Return", "Roll_Vol", "Log_VIX"]
daily["Return"]   = np.log(daily["SPX"]).diff()
daily["Roll_Vol"] = daily["Return"].rolling(WINDOW).std() * np.sqrt(TRADING_DAYS)
daily["Log_VIX"]  = np.log(daily["VIX"])
hmm_daily = daily.loc[START:, HMM_INPUTS].dropna()

def fit_best_hmm(X, n_states=3, seeds=range(5)):
    best, best_ll = None, -np.inf
    for s in seeds:
        m = GaussianHMM(n_components=n_states, covariance_type="full",
                        n_iter=300, tol=1e-4, random_state=s)
        m.fit(X)
        ll = m.score(X)
        if ll > best_ll:
            best, best_ll = m, ll
    return best

def hmm_filter(model, X):
    K, T = model.n_components, len(X)
    log_B = np.column_stack([
        stats.multivariate_normal(model.means_[k], model.covars_[k], allow_singular=True).logpdf(X)
        for k in range(K)])
    alpha = np.zeros((T, K))
    a = model.startprob_ * np.exp(log_B[0] - log_B[0].max())
    alpha[0] = a / a.sum()
    for t in range(1, T):
        a = (alpha[t - 1] @ model.transmat_) * np.exp(log_B[t] - log_B[t].max())
        alpha[t] = a / a.sum()
    return alpha

HMM_CACHE = {}
def hmm_features(train_end):
    if train_end in HMM_CACHE:
        return HMM_CACHE[train_end]
    scaler = StandardScaler().fit(hmm_daily.loc[:train_end])
    X_tr   = scaler.transform(hmm_daily.loc[:train_end])
    model  = fit_best_hmm(X_tr)
    order  = np.argsort(model.means_[:, HMM_INPUTS.index("Roll_Vol")])   # Calm < Volatile < Crisis
    filt   = hmm_filter(model, scaler.transform(hmm_daily))[:, order]
    d = pd.DataFrame(filt, index=hmm_daily.index, columns=["HMM_P_Calm", "HMM_P_Volatile", "HMM_P_Crisis"])
    A = model.transmat_[np.ix_(order, order)]
    out = {"weekly": d.resample(FREQ).last(), "daily": d, "A": A,
           "exp_dur": 1 / (1 - np.diag(A))}
    HMM_CACHE[train_end] = out
    return out

def fold_frame(train_end):
    return base.join(hmm_features(train_end)["weekly"][HMM_FEATS])

print("HMM helpers ready.")

In [ ]:
# — Fit HMMs for all folds and show regime durations ———————————
rows = []
for fold_name, fold in FOLDS.items():
    h = hmm_features(fold["train_end"])
    rows.append({"Fold": fold_name, "Train end": fold["train_end"],
                 **{f"E[days] {r}": d for r, d in zip(["Calm", "Volatile", "Crisis"], h["exp_dur"])}})
h = hmm_features(PROPOSAL_TRAIN_END)
rows.append({"Fold": "Proposal (fixed)", "Train end": PROPOSAL_TRAIN_END,
             **{f"E[days] {r}": d for r, d in zip(["Calm", "Volatile", "Crisis"], h["exp_dur"])}})
print("=== Expected regime duration (trading days) by training window ===")
display(pd.DataFrame(rows).set_index("Fold").round(1))

In [ ]:
# — Visualise filtered regimes (model trained to 2020) —————————
h = hmm_features(FOLDS["Rates-2022"]["train_end"])["daily"]
reg = h.idxmax(axis=1).str.replace("HMM_P_", "")
px  = daily["SPX"].reindex(h.index)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
fig.suptitle("Layer 2 — HMM filtered regimes (trained to 2020-12-31)", fontsize=13, fontweight="bold")
ax = axes[0]
ax.plot(px.index, px, color="#1f4e79", lw=1)
for r, c in colour_map.items():
    ax.fill_between(px.index, 0, 1, where=reg.eq(r), color=c, alpha=0.25,
                    transform=ax.get_xaxis_transform())
ax.set_yscale("log")
ax.set_ylabel("S&P 500 (log)")
ax.legend(handles=[mpatches.Patch(color=c, alpha=0.5, label=r) for r, c in colour_map.items()],
          loc="upper left", fontsize=8)
ax.grid(alpha=0.25)
ax = axes[1]
ax.stackplot(h.index, h["HMM_P_Calm"], h["HMM_P_Volatile"], h["HMM_P_Crisis"],
             colors=list(colour_map.values()), alpha=0.8, labels=["P(Calm)", "P(Volatile)", "P(Crisis)"])
ax.set_ylabel("Probability")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## Part 6 — Layer 3: Expanding Walk-Forward

ทุก fold × horizon:
1. Train = `TRAIN_START` → `train_end` − H สัปดาห์ (purge), ใช้เฉพาะสัปดาห์ที่ไม่ถูก mask
2. **Threshold แบบ alert budget**: สร้าง OOF probability ใน training ด้วย inner TimeSeriesSplit (4 splits, gap = H)
   แล้วตั้ง threshold ที่ quantile $1 - $ `ALERT_BUDGET` → เตือน **20% ของสัปดาห์ที่เสี่ยงที่สุด**
   *(แทน recall target เดิม ซึ่งทำให้โมเดลที่อ่อนต้องลด threshold จนเตือนแทบทุกสัปดาห์ → Recall = 1, Kappa = 0)*
3. Fit บน training ทั้งหมด → ทำนาย test และรายงาน `Alert rate` ที่เกิดขึ้นจริงใน test (อาจไม่ใช่ 20% พอดี เพราะ distribution ของ score เปลี่ยนได้)
4. **AUC พร้อม 95% CI** จาก moving-block bootstrap (block = max(13, H) สัปดาห์ เพราะ label ของสัปดาห์ติดกันซ้อนกัน H สัปดาห์)
   → `vs 0.5`: `> 0.5` ดีกว่าสุ่ม | `≈ 0.5` ไม่ต่างจากสุ่ม | `< 0.5 (reversed)` กลับทิศอย่างมีนัยสำคัญ
5. Models: **RandomForest** และ **Logit** (16 features) กับ **Logit5** (5 features ที่มีทฤษฎีรองรับ)
6. Benchmark ที่ไม่ต้อง train: VIX > 25, yield curve inverted (T10Y2Y < 0), P/D bubble flag

In [ ]:
# — Helpers ————————————————————————————————————————————————————
from scipy.stats import rankdata

MODELS = ["RandomForest", "Logit", "Logit5"]

def make_model(name):
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=400, max_depth=4, min_samples_leaf=20,
                                      max_features="sqrt", class_weight="balanced_subsample",
                                      random_state=SEED, n_jobs=-1)
    C = 1.0 if name == "Logit5" else 0.3          # few features -> lighter regularisation
    return make_pipeline(StandardScaler(),
                         LogisticRegression(class_weight="balanced", C=C, max_iter=3000))

def model_feats(name, feats):
    return SMALL_FEATS if name == "Logit5" else feats

def split(df, train_end, test, h_weeks, label, feats):
    data = df.loc[TRAIN_START:, feats + [label]].dropna()
    tr_end = pd.Timestamp(train_end) - pd.Timedelta(weeks=h_weeks)
    tr, te = data.loc[:tr_end], data.loc[test[0]:test[1]]
    return tr[feats], tr[label].astype(int), te[feats], te[label].astype(int)

LABEL_MODES = {   # how weeks already in a crisis (In_Crisis = 1) are treated
    "all":        "รวมช่วงวิกฤต: ไม่ตัดสัปดาห์ใดเลย (train และ test)",
    "mask":       "ตัดช่วงวิกฤต: ตัดทั้ง train และ test (ค่าหลักของ v2)",
    "train_only": "ตัดเฉพาะตอน train, test ยังครบทุกสัปดาห์",
}

def split_mode(df, train_end, test, h_weeks, name, feats, mode="mask"):
    d = df.loc[TRAIN_START:].copy()
    d["_y"] = d[f"Crisis_{name}"]
    d["_m"] = d["In_Crisis"].eq(1)
    if mode == "mask":
        d["_y"] = d["_y"].where(~d["_m"])
    d = d.dropna(subset=feats + ["_y"])
    tr_end = pd.Timestamp(train_end) - pd.Timedelta(weeks=h_weeks)
    tr, te = d.loc[:tr_end], d.loc[test[0]:test[1]]
    if mode in ("mask", "train_only"):
        tr = tr[~tr["_m"]]
    if mode == "mask":
        te = te[~te["_m"]]
    return tr[feats], tr["_y"].astype(int), te[feats], te["_y"].astype(int)

def pick_threshold(name, X, y, h_weeks, budget=ALERT_BUDGET):
    # Alert budget: threshold = (1 - budget) quantile of out-of-fold scores in the training window
    oof = pd.Series(np.nan, index=y.index)
    # 4 inner splits when possible; fewer when the purge gap (52 weeks for 12M) leaves too few rows
    splits = []
    for k in (4, 3, 2):
        try:
            splits = list(TimeSeriesSplit(n_splits=k, gap=h_weeks).split(X))
            break
        except ValueError:
            continue
    for a, b in splits:
        if y.iloc[a].nunique() < 2:
            continue
        m = make_model(name).fit(X.iloc[a], y.iloc[a])
        oof.iloc[b] = m.predict_proba(X.iloc[b])[:, 1]
    s = oof.dropna()
    return float(np.quantile(s, 1 - budget)) if len(s) >= 20 else 0.5

# — AUC with block-bootstrap confidence intervals ———————————————
def fast_auc(y, s):
    y = np.asarray(y, dtype=int)
    n1 = y.sum(); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(np.asarray(s, dtype=float))
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def block_len(h_weeks):
    return max(13, h_weeks)

def block_indices(n, block, n_boot, seed=SEED):
    block = int(max(1, min(block, n // 2)))
    nb = int(np.ceil(n / block))
    starts = np.random.default_rng(seed).integers(0, n - block + 1, size=(n_boot, nb))
    return (starts[:, :, None] + np.arange(block)).reshape(n_boot, -1)[:, :n]

def auc_ci(y, s, block, n_boot=N_BOOT_AUC):
    # percentile CI; resamples that contain only one class are dropped
    y, s = np.asarray(y, dtype=int), np.asarray(s, dtype=float)
    if len(np.unique(y)) < 2:
        return np.nan, np.nan
    idx = block_indices(len(y), block, n_boot)
    yb, rb = y[idx], rankdata(s[idx], axis=1)
    n1 = yb.sum(axis=1); n0 = yb.shape[1] - n1
    ok = (n1 > 0) & (n0 > 0)
    if ok.sum() < 200:
        return np.nan, np.nan
    auc = ((rb * yb).sum(axis=1)[ok] - n1[ok] * (n1[ok] + 1) / 2) / (n1[ok] * n0[ok])
    lo, hi = np.percentile(auc, [2.5, 97.5])
    return lo, hi

def auc_verdict(lo, hi):
    if np.isnan(lo):
        return "n/a"
    if lo > 0.5:
        return "> 0.5"
    if hi < 0.5:
        return "< 0.5 (reversed)"
    return "≈ 0.5"

def metrics(y, score, thr, h_weeks):
    y = pd.Series(y).astype(int)
    score = pd.Series(score, index=y.index).astype(float)
    yhat = (score >= thr).astype(int)
    both = y.nunique() == 2
    lo, hi = auc_ci(y, score, block_len(h_weeks)) if both else (np.nan, np.nan)
    return {"N test": len(y), "Pos": int(y.sum()), "Base rate": y.mean(),
            "AUC": roc_auc_score(y, score) if both else np.nan,
            "AUC lo": lo, "AUC hi": hi, "vs 0.5": auc_verdict(lo, hi),
            "PR-AUC": average_precision_score(y, score) if y.sum() > 0 else np.nan,
            "Thr": thr, "Alert rate": yhat.mean(),
            "Recall": recall_score(y, yhat, zero_division=0),
            "Precision": precision_score(y, yhat, zero_division=0),
            "Kappa": cohen_kappa_score(y, yhat) if both else np.nan}

BENCHMARKS = {   # name: (score column, sign, rule threshold on signed score)
    "Rule: VIX > 25":        ("VIX", 1, 25),
    "Rule: curve inverted":  ("T10Y2Y", -1, 0),
    "Rule: P/D bubble flag": ("PD_Bubble_Up", 1, 1),
}
print("Helpers ready.")

In [ ]:
# — Run the walk-forward ——————————————————————————————————————
def run_walk_forward(folds, feats, models=tuple(MODELS), benchmarks=True,
                     tune_threshold=True, train_end_override=None, label_mode="mask"):
    res, preds, fitted = [], {}, {}
    need = list(dict.fromkeys(feats + [f for m in models for f in model_feats(m, feats)]))
    for fold_name, fold in folds.items():
        train_end = train_end_override or fold["train_end"]
        df = fold_frame(train_end)
        for h_name, h in HORIZONS.items():
            X_tr, y_tr, X_te, y_te = split_mode(df, train_end, fold["test"], h, h_name, need, label_mode)
            out = pd.DataFrame({"y": y_te})
            for m in models:
                fm = model_feats(m, feats)
                thr = pick_threshold(m, X_tr[fm], y_tr, h) if tune_threshold else 0.5
                model = make_model(m).fit(X_tr[fm], y_tr)
                p = model.predict_proba(X_te[fm])[:, 1]
                out[m], out[f"{m}_thr"] = p, thr
                fitted[(fold_name, h_name, m)] = model
                res.append({"Fold": fold_name, "Horizon": h_name, "Model": m,
                            "N train": len(y_tr), **metrics(y_te, p, thr, h)})
            if benchmarks:
                for b, (col, sign, rthr) in BENCHMARKS.items():
                    s = sign * df.loc[y_te.index, col].astype(float)
                    out[b] = s
                    res.append({"Fold": fold_name, "Horizon": h_name, "Model": b,
                                "N train": 0, **metrics(y_te, s, rthr, h)})
            preds[(fold_name, h_name)] = out
    return pd.DataFrame(res), preds, fitted

res_df, preds, fitted = run_walk_forward(FOLDS, FEATURES)

SHOW = ["N test", "Pos", "Base rate", "AUC", "AUC lo", "AUC hi", "vs 0.5",
        "Alert rate", "Recall", "Precision", "Kappa"]
print(f"=== Main folds (proposal test periods) — alert budget {ALERT_BUDGET:.0%} ===")
main = res_df[res_df["Fold"].isin(MAIN_FOLDS) & res_df["Model"].isin(MODELS)]
display(main.set_index(["Fold", "Horizon", "Model"])[SHOW].round(3))

### 6.1 Results — skill มาจากการเตือนล่วงหน้า หรือจากความต่อเนื่องของวิกฤต?

รัน walk-forward เดียวกัน ต่างกันแค่การจัดการสัปดาห์ที่วิกฤตเกิดไปแล้ว (`In_Crisis` = ต่ำกว่า 52-week high ≥ 10% หรือ PSY crash flag)

| แบบ | Train | Test | วัดอะไร |
|---|---|---|---|
| `all` ไม่ตัด | ทุกสัปดาห์ | ทุกสัปดาห์ | skill รวม = เตือนล่วงหน้า + ทายว่าวิกฤตที่เกิดแล้วจะไปต่อ |
| `mask` ตัด | ตัด | ตัด | early warning จริง |
| `train_only` | ตัด | ทุกสัปดาห์ | โมเดลเดียวกับ `mask` แต่วัดบนทุกสัปดาห์ |

- **Δ persistence = AUC(all) − AUC(mask)** = ส่วนของ AUC ที่มาจากสัปดาห์ที่วิกฤตเกิดแล้ว
- **Share from persistence = Δ / (AUC(all) − 0.5)** = สัดส่วนของ skill เหนือการสุ่มที่มาจากความต่อเนื่องของวิกฤต
  (แสดงเฉพาะเมื่อ CI ของ AUC(all) อยู่เหนือ 0.5 ทั้งช่วง; ค่าเกิน 1 แปลว่าเมื่อตัดช่วงวิกฤตแล้ว AUC ต่ำกว่า 0.5, ค่าติดลบแปลว่าตัดแล้วดีขึ้น)
- CI เป็น **paired block bootstrap** บน predictions รวม 6 fold: แต่ละรอบ resample ชุดสัปดาห์เดียวกันให้ทุกแบบ

In [ ]:
# — Results table: not masked vs masked, pooled over all folds ————
mode_res, mode_preds = {"mask": res_df}, {"mask": preds}     # main run = masked
for mode in ["all", "train_only"]:
    r, p, _ = run_walk_forward(FOLDS, FEATURES, benchmarks=False, label_mode=mode)
    mode_res[mode], mode_preds[mode] = r, p

def pooled_frame(h_name, m):
    J = pd.concat([mode_preds["all"][(f, h_name)] for f in FOLDS])[["y", m]].rename(columns={m: "s_all"})
    J["s_tr"]   = pd.concat([mode_preds["train_only"][(f, h_name)] for f in FOLDS])[m].reindex(J.index)
    J["s_mask"] = pd.concat([mode_preds["mask"][(f, h_name)] for f in FOLDS])[m].reindex(J.index)
    return J

def paired_boot(J, block, n_boot=N_BOOT_AUC):
    y, sa, st, sm = (J[c].values for c in ["y", "s_all", "s_tr", "s_mask"])
    y = y.astype(int)
    keep = ~np.isnan(sm)
    out = np.full((n_boot, 3), np.nan)
    for i, ix in enumerate(block_indices(len(J), block, n_boot)):
        k = keep[ix]
        out[i] = fast_auc(y[ix], sa[ix]), fast_auc(y[ix][k], sm[ix][k]), fast_auc(y[ix], st[ix])
    return out[~np.isnan(out).any(axis=1)]

rows = []
for h_name, h in HORIZONS.items():
    for m in MODELS:
        J = pooled_frame(h_name, m)
        keep = J["s_mask"].notna()
        a_all  = fast_auc(J["y"], J["s_all"])
        a_mask = fast_auc(J.loc[keep, "y"], J.loc[keep, "s_mask"])
        a_tr   = fast_auc(J["y"], J["s_tr"])
        B = paired_boot(J, block_len(h))
        (alo, ahi), (mlo, mhi), (dlo, dhi) = (np.percentile(v, [2.5, 97.5])
                                              for v in (B[:, 0], B[:, 1], B[:, 0] - B[:, 1]))
        rows.append({"Horizon": h_name, "Model": m,
                     "Weeks all": len(J), "Weeks mask": int(keep.sum()),
                     "Pos all": int(J["y"].sum()), "Pos mask": int(J.loc[keep, "y"].sum()),
                     "AUC all": a_all, "all lo": alo, "all hi": ahi, "all vs 0.5": auc_verdict(alo, ahi),
                     "AUC mask": a_mask, "mask lo": mlo, "mask hi": mhi, "mask vs 0.5": auc_verdict(mlo, mhi),
                     "AUC train_only": a_tr,
                     "Δ persistence": a_all - a_mask, "Δ lo": dlo, "Δ hi": dhi,
                     "Share from persistence": (a_all - a_mask) / (a_all - 0.5) if alo > 0.5 else np.nan})
results = pd.DataFrame(rows).set_index(["Horizon", "Model"])

for mode, desc in LABEL_MODES.items():
    print(f"{mode:11s} = {desc}")
print("=" * 70)
print("=== RESULTS: pooled AUC, not masked vs masked (95% paired block-bootstrap CI) ===")
display(results.round(3))

# Operating metrics at the alert budget, averaged over folds
idx = pd.MultiIndex.from_product([HORIZONS, MODELS], names=["Horizon", "Model"])
ops = pd.concat({mode: r[r["Model"].isin(MODELS)]
                       .groupby(["Horizon", "Model"])[["Alert rate", "Recall", "Precision", "Kappa"]]
                       .mean().reindex(idx)
                 for mode, r in mode_res.items()}, axis=1)
print(f"=== Mean fold metrics at alert budget {ALERT_BUDGET:.0%} ===")
display(ops.round(3))

In [ ]:
# — Visual: AUC not masked vs masked, with 95% CI ——————————————————
fig, axes = plt.subplots(1, len(MODELS), figsize=(15, 4.5), sharey=True)
x = np.arange(len(HORIZONS))
for ax, m in zip(axes, MODELS):
    r = results.xs(m, level="Model").reindex(list(HORIZONS))
    for off, (col, lo, hi, lab, c) in zip([-0.15, 0.15],
            [("AUC all", "all lo", "all hi", "Not masked (all)", "#95a5a6"),
             ("AUC mask", "mask lo", "mask hi", "Masked (early warning)", "#e74c3c")]):
        ax.errorbar(x + off, r[col], yerr=[r[col] - r[lo], r[hi] - r[col]],
                    fmt="o", color=c, capsize=4, lw=1.5, label=lab)
    ax.axhline(0.5, color="black", ls="--", lw=1)
    ax.set_xticks(x)
    ax.set_xticklabels(list(HORIZONS))
    ax.set(title=m, xlabel="Horizon", ylim=(0, 1))
    ax.grid(alpha=0.3, axis="y")
axes[0].set_ylabel("Pooled AUC")
axes[0].legend(fontsize=8, loc="lower left")
fig.suptitle("Results — how much skill comes from crisis persistence?", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# — All folds: AUC and test against 0.5 ————————————————————————
full_idx = pd.MultiIndex.from_product([FOLDS, HORIZONS])
auc_all = res_df.pivot_table(index=["Fold", "Horizon"], columns="Model", values="AUC")
auc_all = auc_all[MODELS + list(BENCHMARKS)].reindex(full_idx)
print("=== AUC — every fold and horizon ===")
display(auc_all.round(3))

verdict = res_df.pivot_table(index=["Fold", "Horizon"], columns="Model", values="vs 0.5",
                             aggfunc="first")[MODELS + list(BENCHMARKS)].reindex(full_idx)
print("=== Is each fold's AUC different from 0.5? (95% block-bootstrap CI) ===")
display(verdict.fillna("n/a"))

# Pooled out-of-sample AUC (test windows do not overlap -> one pseudo real-time series)
pooled_rows, ci_rows = [], []
for h_name, h in HORIZONS.items():
    stack = pd.concat([preds[(f, h_name)] for f in FOLDS])
    row = {"Horizon": h_name, "N": len(stack), "Base rate": stack["y"].mean()}
    for m in MODELS + list(BENCHMARKS):
        row[m] = roc_auc_score(stack["y"], stack[m]) if stack["y"].nunique() == 2 else np.nan
        lo, hi = auc_ci(stack["y"], stack[m], block_len(h))
        ci_rows.append({"Horizon": h_name, "Model": m, "AUC": row[m],
                        "95% CI low": lo, "95% CI high": hi, "vs 0.5": auc_verdict(lo, hi)})
    pooled_rows.append(row)
pooled = pd.DataFrame(pooled_rows).set_index("Horizon")
pooled_ci = pd.DataFrame(ci_rows).set_index(["Horizon", "Model"])
print("=== Pooled out-of-sample AUC (all 6 folds stacked, masked) ===")
display(pooled.round(3))
print("=== Pooled AUC with 95% block-bootstrap CI ===")
display(pooled_ci.round(3))

In [ ]:
# — Visual: AUC per fold, 3M horizon ——————————————————————————
H_PLOT = "3M"
tbl = auc_all.xs(H_PLOT, level=1)
fig, ax = plt.subplots(figsize=(13, 4.5))
tbl.plot(kind="bar", ax=ax, color=["#e74c3c", "#1f4e79", "#27ae60", "#95a5a6", "#bdc3c7", "#d7bde2"],
         alpha=0.9, width=0.85)
ax.axhline(0.5, color="black", ls="--", lw=1)
ax.set(title=f"Out-of-sample AUC by fold ({H_PLOT} horizon)", ylabel="AUC", xlabel="", ylim=(0, 1))
ax.tick_params(axis="x", rotation=0)
ax.legend(fontsize=8, ncol=3)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# — Probability paths for the main folds ——————————————————————
fig, axes = plt.subplots(2, 2, figsize=(15, 7), sharex="col",
                         gridspec_kw={"height_ratios": [1, 1.2]})
fig.suptitle(f"Out-of-sample P(crisis within {H_PLOT}) — masked weeks shown in grey",
             fontsize=13, fontweight="bold")
for col, fold_name in enumerate(MAIN_FOLDS):
    t0, t1 = FOLDS[fold_name]["test"]
    wk   = base.loc[t0:t1]
    out  = preds[(fold_name, H_PLOT)].reindex(wk.index)      # NaN in masked weeks -> gaps in lines
    ax = axes[0, col]
    ax.plot(wk.index, wk["SPX"], color="#1f4e79", lw=1.2)
    ax.fill_between(wk.index, 0, 1, where=wk["In_Crisis"].eq(1), color="gray", alpha=0.3,
                    transform=ax.get_xaxis_transform(), label="Masked (crisis under way)")
    ax.set(title=fold_name, ylabel="S&P 500")
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(alpha=0.3)
    ax = axes[1, col]
    ax.fill_between(out.index, 0, 1, where=out["y"].eq(1), color="#e74c3c", alpha=0.15,
                    transform=ax.get_xaxis_transform(), label="Actual label = 1")
    ax.plot(out.index, out["RandomForest"], color="#e74c3c", lw=1.5, label="Random Forest")
    ax.plot(out.index, out["Logit"], color="#1f4e79", lw=1.2, ls="--", label="Logit")
    ax.plot(out.index, out["Logit5"], color="#27ae60", lw=1.2, label="Logit5")
    ax.axhline(out["RandomForest_thr"].dropna().iloc[0], color="#e74c3c", ls=":", lw=1, label="RF threshold")
    ax.set(ylabel="P(crisis)", ylim=(0, 1))
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.tight_layout()
plt.show()

## Part 7 — Statistical Tests (block bootstrap on pooled predictions)

ใช้ predictions ของทั้ง 6 fold ต่อกัน (test window ไม่ทับกัน) → sample ใหญ่ขึ้น ทดสอบได้มีพลังกว่าทีละ fold
Moving-block bootstrap (block = max(13, H) สัปดาห์) รักษา autocorrelation ภายใน block

In [ ]:
# — Block bootstrap AUC difference ————————————————————————————
def block_bootstrap_auc_diff(y, s1, s2, block=13, n_boot=2000, seed=SEED):
    y, s1, s2 = (np.asarray(v, dtype=float) for v in (y, s1, s2))
    diffs = np.array([fast_auc(y[ix], s1[ix]) - fast_auc(y[ix], s2[ix])
                      for ix in block_indices(len(y), block, n_boot, seed)])
    diffs = diffs[~np.isnan(diffs)]
    return diffs.mean(), *np.percentile(diffs, [2.5, 97.5]), (diffs <= 0).mean()

COMPARISONS = [("RandomForest", "Logit"),
               ("Logit5", "RandomForest"),
               ("RandomForest", "Rule: VIX > 25"),
               ("RandomForest", "Rule: curve inverted"),
               ("Logit5", "Rule: curve inverted")]
rows = []
for h_name, h in HORIZONS.items():
    stack = pd.concat([preds[(f, h_name)] for f in FOLDS])
    for a, b in COMPARISONS:
        d, lo, hi, pv = block_bootstrap_auc_diff(stack["y"], stack[a], stack[b], block=block_len(h))
        rows.append({"Horizon": h_name, "Comparison": f"{a} − {b}", "AUC diff": d,
                     "95% CI low": lo, "95% CI high": hi, "p (one-sided)": pv})
print("=== Pooled block-bootstrap tests ===")
display(pd.DataFrame(rows).round(3))
print("CI entirely above 0 -> the first model is significantly better.")

## Part 8 — Ablation Studies

1. **Feature sets**: Full vs ไม่มี PSY feature vs Full + weekly price bubble (ตรวจซ้ำว่า price BSADF เป็น noise ไหม)
2. **Training scheme**: Fixed 1997–2008 (ตาม proposal) vs Expanding — บน 2 fold หลัก

In [ ]:
# — 8.1 Feature-set ablation (RandomForest, all folds) —————————
FEATURE_SETS = {
    "Full":                   FEATURES,
    "No PSY":                 [f for f in FEATURES if f not in PSY_FEATS],
    "Full + price bubble":    FEATURES + PRICE_PSY,
}
abl_preds, abl_rows = {}, []
for set_name, feats in FEATURE_SETS.items():
    r, p, _ = run_walk_forward(FOLDS, feats, models=("RandomForest",), benchmarks=False,
                               tune_threshold=False)
    abl_preds[set_name] = p
    for h_name in HORIZONS:
        stack = pd.concat([p[(f, h_name)] for f in FOLDS])
        abl_rows.append({"Feature set": set_name, "Horizon": h_name,
                         "Pooled AUC": roc_auc_score(stack["y"], stack["RandomForest"]),
                         "Pooled PR-AUC": average_precision_score(stack["y"], stack["RandomForest"]),
                         "Mean fold AUC": r["AUC"].groupby(r["Horizon"]).mean()[h_name]})
abl = pd.DataFrame(abl_rows)
print("=== Feature-set ablation ===")
display(abl.pivot_table(index="Horizon", columns="Feature set", values="Pooled AUC")
        [list(FEATURE_SETS)].round(3))

rows = []
for h_name in HORIZONS:
    full = pd.concat([abl_preds["Full"][(f, h_name)] for f in FOLDS])
    for other in ["No PSY", "Full + price bubble"]:
        oth = pd.concat([abl_preds[other][(f, h_name)] for f in FOLDS])
        d, lo, hi, pv = block_bootstrap_auc_diff(full["y"], full["RandomForest"], oth["RandomForest"])
        rows.append({"Horizon": h_name, "Comparison": f"Full − {other}", "AUC diff": d,
                     "95% CI low": lo, "95% CI high": hi, "p (one-sided)": pv})
print("=== Does each change matter? (pooled block bootstrap) ===")
display(pd.DataFrame(rows).round(3))

In [ ]:
# — 8.2 Training-scheme ablation (main folds) ——————————————————
main_folds = {k: FOLDS[k] for k in MAIN_FOLDS}
fixed_res, _, _ = run_walk_forward(main_folds, FEATURES, models=("RandomForest",), benchmarks=False,
                                   tune_threshold=False, train_end_override=PROPOSAL_TRAIN_END)
exp_res, _, _   = run_walk_forward(main_folds, FEATURES, models=("RandomForest",), benchmarks=False,
                                   tune_threshold=False)
scheme = pd.DataFrame({
    "Fixed 1997–2008 (proposal)": fixed_res.set_index(["Fold", "Horizon"])["AUC"],
    "Expanding": exp_res.set_index(["Fold", "Horizon"])["AUC"],
})
scheme["Change"] = scheme["Expanding"] - scheme["Fixed 1997–2008 (proposal)"]
print("=== RandomForest AUC: fixed vs expanding training window ===")
display(scheme.round(3))

## Part 9 — Why random splits mislead (leakage check)

Başoğlu Kabran & Ünlü (2021) รายงาน accuracy ~99% จากการแบ่ง train/test แบบสุ่ม
เทียบให้เห็นว่าโมเดลเดียวกัน ข้อมูลเดียวกัน ถ้าแบ่งแบบสุ่มจะได้ AUC สูงเกินจริงแค่ไหน

In [ ]:
# — Random 80/20 split vs walk-forward ————————————————————————
df_all = fold_frame(FOLDS["Rates-2022"]["train_end"])
rows = []
for h_name in HORIZONS:
    data = df_all.loc[TRAIN_START:, FEATURES + [f"Label_{h_name}"]].dropna()
    X, y = data[FEATURES], data[f"Label_{h_name}"].astype(int)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    p = make_model("RandomForest").fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
    rows.append({"Horizon": h_name, "Random split AUC": roc_auc_score(y_te, p),
                 "Walk-forward pooled AUC": pooled.loc[h_name, "RandomForest"]})
leak = pd.DataFrame(rows).set_index("Horizon")
leak["Inflation"] = leak["Random split AUC"] - leak["Walk-forward pooled AUC"]
print("=== Leakage from random splitting ===")
display(leak.round(3))
print("Neighbouring weeks are nearly identical, so a random split lets the model 'remember' the test set.")

## Part 10 — Explainability (SHAP)

In [ ]:
# — SHAP for the 3M RandomForest in the main folds ————————————
def rf_shap(model, X):
    sv = shap.TreeExplainer(model).shap_values(X)
    if isinstance(sv, list):
        sv = sv[1]
    elif np.ndim(sv) == 3:
        sv = sv[:, :, 1]
    return sv

imp = {}
for fold_name in MAIN_FOLDS:
    fold = FOLDS[fold_name]
    df = fold_frame(fold["train_end"])
    _, _, X_te, _ = split(df, fold["train_end"], fold["test"], HORIZONS[H_PLOT], f"Label_{H_PLOT}", FEATURES)
    sv = rf_shap(fitted[(fold_name, H_PLOT, "RandomForest")], X_te)
    imp[fold_name] = pd.Series(np.abs(sv).mean(axis=0), index=FEATURES)
    shap.summary_plot(sv, X_te, show=False, max_display=12)
    plt.title(f"SHAP summary — {fold_name} ({H_PLOT})", fontsize=12)
    plt.tight_layout()
    plt.show()

imp = pd.DataFrame(imp)
imp = imp / imp.sum()
groups = {"Bubble & credit PSY": PSY_FEATS, "HMM regime": HMM_FEATS,
          "Market": MARKET_FEATS, "Rates & macro": MACRO_FEATS}
grp = pd.DataFrame({g: imp.loc[f].sum() for g, f in groups.items()}).T

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
imp.sort_values(MAIN_FOLDS[0]).plot(kind="barh", ax=axes[0], color=["#e74c3c", "#1f4e79"], alpha=0.85)
axes[0].set(title="Share of |SHAP| by feature", xlabel="Share")
grp.plot(kind="barh", ax=axes[1], color=["#e74c3c", "#1f4e79"], alpha=0.85)
axes[1].set(title="Share of |SHAP| by feature group", xlabel="Share")
for ax in axes:
    ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()
display(grp.round(3))

## Part 11 — Two System Outputs

1. **Early warning** (Random Forest): alert = P ≥ threshold ของ fold นั้น ติดต่อกัน 2 สัปดาห์ → lead time ก่อน peak
2. **Real-time crash detection** (weekly PSY, Phillips & Shi 2020): ยืนยันว่าวิกฤตเริ่มแล้วเร็วแค่ไหนหลัง peak

In [ ]:
# — Early-warning lead time and crash-detection delay —————————
def alerts(p, thr, k=ALERT_K):
    return (p >= thr).astype(int).rolling(k).sum().eq(k)

rows = []
for fold_name, fold in FOLDS.items():
    t0, t1 = fold["test"]
    spx_t  = base.loc[t0:t1, "SPX"]
    dd     = spx_t / spx_t.cummax() - 1
    trough = dd.idxmin()
    peak   = spx_t.loc[:trough].idxmax()

    out = preds[(fold_name, H_PLOT)]
    a   = alerts(out["RandomForest"], out["RandomForest_thr"].iloc[0])
    win = a.loc[peak - pd.Timedelta(weeks=26):peak]
    first_alert = win[win].index.min() if win.any() else pd.NaT
    false_share = (a & out["y"].eq(0)).sum() / max(a.sum(), 1)

    cf = base.loc[peak:trough + pd.Timedelta(weeks=26), "Crash_Flag"]
    first_crash = cf[cf.eq(1)].index.min() if cf.eq(1).any() else pd.NaT
    dd_at_detect = (base.loc[first_crash, "SPX"] / base.loc[peak, "SPX"] - 1) if pd.notna(first_crash) else np.nan

    rows.append({"Fold": fold_name, "Peak": peak.date(), "Trough": trough.date(),
                 "Max DD": f"{dd.min():.1%}",
                 "First EW alert": first_alert.date() if pd.notna(first_alert) else "none",
                 "EW lead (weeks)": (peak - first_alert).days / 7 if pd.notna(first_alert) else np.nan,
                 "EW false-alarm share": round(false_share, 2),
                 "Crash detected": first_crash.date() if pd.notna(first_crash) else "none",
                 "Detect delay (weeks)": (first_crash - peak).days / 7 if pd.notna(first_crash) else np.nan,
                 "DD at detection": f"{dd_at_detect:.1%}" if pd.notna(dd_at_detect) else "—"})

print(f"=== Output 1: early warning (H = {H_PLOT}) | Output 2: real-time crash detection ===")
display(pd.DataFrame(rows).set_index("Fold"))

calm = base["DD_52w"] > -0.05
print(f"Crash flags raised while the market was within 5% of its 52-week high: "
      f"{int((base['Crash_Flag'].eq(1) & calm).sum())} weeks (false crash signals)")

## Part 12 — Summary

In [ ]:
# — Summary for the report ————————————————————————————————————
print("=" * 70)
print(f"Layer 1 — PSY ({'BIC lag' if FORCE_LAG is None else f'lag forced to {FORCE_LAG}'} + wild bootstrap)")
print(f"  P/D monthly   : GSADF {pd_info['gsadf']:.2f} vs CV {pd_info['cv']:.2f} (lag {pd_info['lag']})")
print(f"  Credit weekly : GSADF {cr_info['gsadf']:.2f} vs CV {cr_info['cv']:.2f} (lag {cr_info['lag']})")
print(f"  Price weekly  : GSADF {px_info['gsadf']:.2f} vs CV {px_info['cv']:.2f} (lag {px_info['lag']})")
print(f"  Replication of Phillips & Shi (2020): {hit:.0%} of paper-flagged months matched")
print("Layer 3 — pooled out-of-sample AUC (6 folds)")
print("=" * 70)
print("Replication — BIC lag vs lag 0:")
display(val_cmp[["Lag used", "Paper months matched", "1998–2000 matched", "1998–2000 extra"]].round(3))
print("RESULTS — pooled AUC, not masked vs masked:")
display(results[["AUC all", "all vs 0.5", "AUC mask", "mask vs 0.5", "Δ persistence", "Δ lo", "Δ hi",
                 "Share from persistence"]].round(3))
print("Pooled AUC (masked) with 95% CI:")
display(pooled_ci.round(3))
print(f"Main folds (proposal test periods), alert budget {ALERT_BUDGET:.0%}:")
display(main.set_index(["Fold", "Horizon", "Model"])[SHOW].round(3))

### Notes
- ทุกตัวเลขเป็น **out-of-sample** จาก expanding walk-forward; threshold และ critical value เลือกจากข้อมูลก่อน test เท่านั้น
- ผลจาก test ที่ mask แล้ววัดเฉพาะ "สัปดาห์ที่วิกฤตยังไม่เริ่ม" ซึ่งเป็นโจทย์ของ early warning จริงๆ
- ส่วนต่างระหว่างโมเดลให้ดูจาก Part 7–8 (bootstrap CI) ก่อนสรุปว่าดีกว่ากัน
- AUC ที่ CI คร่อม 0.5 ให้รายงานว่า "ไม่ต่างจากสุ่ม" ไม่ใช่ "ดีกว่าสุ่มเล็กน้อย"; CI ของ fold เดี่ยวกว้างมากเพราะ test window สั้น (โดยเฉพาะ 12M)
- Alert rate ใน test อาจห่างจาก 20% ได้ เพราะ threshold ตั้งจาก OOF score ของ training
- ข้อจำกัด: lag ของ ADF คงที่ทุก window (paper เลือกใหม่ทุก window), P/D จาก Shiller ต่างจาก Datastream เล็กน้อย, จำนวนวิกฤตในข้อมูลยังน้อย

**References**
- Phillips, P.C.B., Shi, S., & Yu, J. (2015). Testing for multiple bubbles. *International Economic Review*, 56(4), 1043–1078.
- Phillips, P.C.B., & Shi, S. (2020). Real time monitoring of asset markets: Bubbles and crises. In *Handbook of Statistics*, Vol. 42, 61–80. (Cowles Foundation DP 2152, 2018)
- Başoğlu Kabran, F., & Ünlü, K.D. (2021). A two-step machine learning approach to predict S&P 500 bubbles. *Journal of Applied Statistics*, 48(13–15), 2776–2794.
- Brunnermeier, M.K., & Sannikov, Y. (2014). A macroeconomic model with a financial sector. *American Economic Review*, 104(2), 379–421.
- Danielsson, J., Valenzuela, M., & Zer, I. (2018). Learning from history: Volatility and financial crises. *Review of Financial Studies*, 31(7), 2774–2805.
- Estrella, A., & Mishkin, F.S. (1998). Predicting U.S. recessions: Financial variables as leading indicators. *Review of Economics and Statistics*, 80(1), 45–61.
- Gu, X. (2022). *Early-Warning Alert Systems for Financial-Instability Detection: An HMM-Driven Approach*. PhD thesis, Western University.